# RAG Contest — Baseline / ขั้นตอนเดิม 1–7
ใช้ Qwen3-1.7B Q8 CPU ไม่ Fine-tuning โค้ดตรงกับ baseline 161.90/200
ส่วนติดตั้งใช้วิธีเดิมของ RAG_Contest_Colab ไม่ใช้ตัวติดตั้งของ Competition_Baseline_CPU
ยังไม่ได้ยืนยันการรัน Colab ของไฟล์ฉบับนี้ ไม่มีการอ่านเฉลยในเส้นทางตอบ
เลือก CPU และรันทีละเซลล์ ข้อ 5B ใช้รันต่อใน runtime เดิมได้

## 1. เชื่อม Drive และตั้งตำแหน่ง
ใส่ PDF ทั้งหมดและคำถามที่จะตอบ ค่า EXPECTED_COUNT ต้องตรงจำนวนคำถามจริง
OUTPUT ใช้โฟลเดอร์ rag_results เดิม เพื่อพบโมเดลที่เคยดาวน์โหลดไว้
เมื่อ runtime ถูกล้าง: ตั้ง RUN_MODE="resume" และ RESUME_RUN="latest" แล้วรันข้อ 1–5B ตามลำดับ
ถ้ารอบเก่าสร้างก่อนมี pointer ให้ใส่ path รอบเดิมแทน latest

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import time, sys, subprocess, json, os
DOCS = Path("/content/drive/MyDrive/NLP Project/sample_docs")
QUESTIONS = Path("/content/drive/MyDrive/NLP Project/question_samples_test.csv")
OUTPUT = Path("/content/drive/MyDrive/NLP Project/competition_results")
DEVICE = "cpu"
EXPECTED_COUNT = 100
RUN_MODE = "new"  # ครั้งแรก new; runtime หลุด/ถูกล้าง เปลี่ยนเป็น resume
RESUME_RUN = "latest"  # latest = รอบล่าสุดของ notebook นี้ หรือใส่ path ของรอบเดิม
assert DOCS.is_dir(), f"ไม่พบ PDF folder: {DOCS}"
if RUN_MODE == "new":
    assert QUESTIONS.is_file(), f"ไม่พบคำถาม: {QUESTIONS}"
OUTPUT.mkdir(parents=True, exist_ok=True)

assert RUN_MODE in ("new", "resume")
LATEST_RUN_FILE = OUTPUT / "baseline_contest_latest.json"
RESUME_PATH = None
if RUN_MODE == "resume":
    if RESUME_RUN.strip() == "latest":
        assert LATEST_RUN_FILE.exists(), "ไม่พบ pointer รอบล่าสุด: ใส่ RESUME_RUN เป็น path รอบเดิม"
        RESUME_PATH = Path(json.loads(LATEST_RUN_FILE.read_text())["run"])
    else:
        assert RESUME_RUN.strip(), "ใส่ RESUME_RUN เป็น latest หรือ path รอบเดิม"
        RESUME_PATH = Path(RESUME_RUN)
    assert (RESUME_PATH/"questions.csv").is_file(), "รอบเดิมไม่มี questions.csv ตรวจ path"
    assert (RESUME_PATH/"config.json").is_file(), "รอบเดิมไม่มี config.json"
    print("จะรันต่อจาก:", RESUME_PATH)


## 2. ติดตั้งแบบ RAG_Contest_Colab เดิม
ใช้ uv และ rag_cpu_venv เดิม แยกพื้นที่โค้ด baseline เพื่อไม่ปะปนกับ v5
ถ้าติดตั้งไม่ผ่านให้หยุดและส่งข้อความ error ไม่ข้ามไปข้อ 5

In [ ]:
SETUP_START = time.perf_counter()
PROJECT = Path("/content/rag_contest_baseline_project")
PROJECT.mkdir(exist_ok=True)
import base64, io, zipfile
PAYLOAD = "..."
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as z:
    for name in z.namelist():
        target = (PROJECT / name).resolve()
        assert target.is_relative_to(PROJECT.resolve())
    z.extractall(PROJECT)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
# Reload this notebook's bundled code instead of keeping earlier imports in the kernel.
for module_name in list(sys.modules):
    if module_name == "rag_contest" or module_name.startswith("rag_contest."):
        del sys.modules[module_name]

# Isolated environment avoids conflicts with Colab's preinstalled GPU/vision packages.
assert DEVICE in ("cpu", "cuda")
if DEVICE == "cuda":
    subprocess.run(["nvidia-smi"], check=True)
VENV = Path("/content/rag_gpu_venv" if DEVICE == "cuda" else "/content/rag_cpu_venv")
from rag_contest.bootstrap import ensure_environment, ensure_uv
UV = ensure_uv(PROJECT / "uv-bin")
PYTHON = ensure_environment(VENV, uv=UV)
from rag_contest.notebook_io import stream_command

def uv_install(*packages):
    # Capture explicitly: some notebook frontends hide subprocess stderr.
    for flag in ("--index-url", "--extra-index-url"):
        if flag in packages:
            from urllib.parse import urlsplit
            url = packages[packages.index(flag) + 1]
            assert urlsplit(url).scheme == "https" and not any(c in url for c in "[]() "), f"ใช้ URL ธรรมดา ไม่ใช่ Markdown: {url!r}"
    command = [UV, "pip", "install", "--python", PYTHON, *packages]
    print("กำลังติดตั้ง:", " ".join(packages), flush=True)
    status = stream_command(command, log_path=PROJECT / "install.log")
    if status:
        raise subprocess.CalledProcessError(status, command)

uv_install("setuptools", "wheel")
uv_install("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
requirements = [line.strip() for line in Path("requirements.txt").read_text().splitlines()
                if line.strip() and not line.startswith("#") and not line.startswith("llama-cpp")]
uv_install(*requirements)
GGUF_AVAILABLE = True
try:
    if DEVICE == "cuda":
        # Build against the runtime CUDA toolkit, including Python versions without prebuilt wheels.
        import shutil
        if not shutil.which("nvcc"):
            raise RuntimeError("ไม่พบ CUDA toolkit: เลือก GPU runtime แล้วเริ่มใหม่")
        old_cmake = os.environ.get("CMAKE_ARGS")
        old_jobs = os.environ.get("CMAKE_BUILD_PARALLEL_LEVEL")
        os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=ON"
        os.environ["CMAKE_BUILD_PARALLEL_LEVEL"] = "2"
        try:
            uv_install("llama-cpp-python>=0.3.9,<0.4", "--no-binary", "llama-cpp-python", "--no-cache")
        finally:
            for key, old in (("CMAKE_ARGS", old_cmake), ("CMAKE_BUILD_PARALLEL_LEVEL", old_jobs)):
                if old is None: os.environ.pop(key, None)
                else: os.environ[key] = old
        status = stream_command([PYTHON, "-c", "from rag_contest.gpu import require_cuda; require_cuda()"])
        if status:
            raise RuntimeError("ตัวรันไม่รองรับ GPU: หยุดก่อนตอบ ส่ง install.log มาตรวจ")
    else:
        uv_install("llama-cpp-python>=0.3.9,<0.4", "--extra-index-url",
                    "https://abetlen.github.io/llama-cpp-python/whl/cpu")
    status = stream_command([PYTHON, "-c", "from llama_cpp import Llama"])
    if status:
        raise subprocess.CalledProcessError(status, [PYTHON, "-c", "from llama_cpp import Llama"])
except subprocess.CalledProcessError:
    if DEVICE == "cuda":
        raise RuntimeError("ติดตั้ง GPU ไม่สำเร็จ ไม่สลับไป CPU อัตโนมัติ ดู install.log")
    GGUF_AVAILABLE = False
    print("ติดตั้ง GGUF ไม่สำเร็จ: ใช้ Transformers สำหรับพัฒนา แล้ววัด RAM/เวลาใหม่")

def cli(*args, allow_incomplete=False):
    try:
        status = stream_command([PYTHON, "-u", "-m", "rag_contest", *map(str, args)])
    except KeyboardInterrupt:
        print("หยุดโปรแกรมแล้ว ข้อที่บันทึก checkpoint สำเร็จยังอยู่ใน Drive", flush=True)
        raise
    if status and not (allow_incomplete and status == 2):
        raise RuntimeError(f"ขั้นตอนหยุดด้วยรหัส {status}; อ่านข้อความด้านบน")
    return status

import hashlib
EXPECTED_SOURCES = {'blocks.py': '051c01c9e6df65e5a37e8182f3b11011c7a5d7277cc24b10a5184802bf7ce7ba', 'bootstrap.py': '78a5c7b0faaebe22ca1cf9e43b2f853862d57fda421dded1d5e73812df9df538', 'calculation.py': '57b84243a33ecc52b2ee17fc08e8ad843507d80fc27ebd6c4582a4121566b081', 'competition.py': 'f421ea7702837fe7a361a217b4bc2d11a670fc93b5afbc6ad7aca05bae2fe1f3', 'data.py': 'e34a0b2e387a0bc47e08a356a12cba6589e59f56bc7e8de71646862deb9d9531', 'evidence.py': '28cd2db3fb0e837a6d41db8525951021c04ed6b0811ee44c2e250ed7fb68e6ac', 'gpu.py': 'f16f45fe94ab15ecbf35fb1e4e2233f9d5a248066be935858260dec0bce2ba9b', 'index_cache.py': 'c53876474baa5d7644b31a2c5d0c1d24c91983e8a2ca70cf30bdcc65e982265c', 'models.py': '74eb3163dc2eb7b981b9a601fa2b1dd5764598060e93a9ddd10cdd9552e58ad9', 'notebook_io.py': 'c59a23135dedda17504bd75f5e4468889682f3238bf0ef9acbeb012e76316ef4', 'pipeline.py': 'a52d5d2097d8f99ec47ffa1f88fafbbf881273d936e7b6398ac1afff6bedf2e0', 'preflight.py': '1d527bcf0c757e74cf01f390da8ecbc73001504cc62dd5c786cb205493673ae3', 'prompts.py': '030255017a4e3cffdc8a49918ae743b4d1ea9efa25f2b7ebd30552edbcc3bda8', 'retrieval.py': '95a4bde7e6ef426c4f9ce8abe7478159b10a49d3b8055e6e8934fa7f91bee40f', 'runtime.py': 'fcf3ac0ebc927ea4180a502a0860abf7d1c61fff10c4e43439865d219651023b', 'trials.py': '3c45daaeb7366c7ad6c2ad4b179e39ce51feaba05feea262afef12b192d6ef64', '__init__.py': '7bdd72c75b57ae6834bf149f902024b4d04d3ec5890f150ee4bbf2a5f226a393', '__main__.py': 'fe342ef03b0c686d56050a6544d116eb975f1dad37d27c70b9c09e412ab1a4b3'}
actual = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in (PROJECT/'rag_contest').glob('*.py')}
assert actual == EXPECTED_SOURCES, 'โค้ด baseline ไม่ตรง หยุดก่อนรัน'
assert GGUF_AVAILABLE, 'ต้องติดตั้ง llama-cpp-python สำเร็จก่อน ใช้ Q8 เท่านั้น'
print('ตรวจโค้ด baseline ผ่าน; ไม่ใช่การยืนยันคะแนนบน Colab')


## 3. ตรวจไฟล์คำถาม
ไม่แบ่ง dev/holdout และไม่อ่านเฉลย ใช้ทุกคำถามในไฟล์ที่ระบุ

In [ ]:
from rag_contest.data import load_questions
question_source = RESUME_PATH/'questions.csv' if RESUME_PATH is not None else QUESTIONS
questions = load_questions(question_source)
assert len(questions) == EXPECTED_COUNT, f"รอบนี้มี {len(questions)} ข้อ แต่ตั้ง EXPECTED_COUNT={EXPECTED_COUNT}"
assert len({r['question_id'] for r in questions}) == len(questions), 'question_id ซ้ำ'
print('คำถาม',len(questions),'ข้อ จาก',question_source)


## 4. เตรียมโมเดลและ config baseline
ใช้ prepare และ stage_assets แบบเดิม ตรวจความครบไฟล์ตาม assets.json
ไม่เทียบเวลาดาวน์โหลดระหว่างเครื่อง ไม่มีการเปลี่ยนโมเดลโดยอัตโนมัติ

In [ ]:
assert GGUF_AVAILABLE, "รอบนี้ใช้ Q8 เท่านั้น: แก้การติดตั้ง GGUF ก่อน ไม่เปลี่ยนเป็น HF อัตโนมัติ"
assert DEVICE == "cpu", "การซ้อมรอบนี้ใช้ CPU"
MODEL = "qwen3-1.7b-q8"
DRIVE_ASSETS = OUTPUT / "assets" / MODEL
cli("prepare", "--model", MODEL, "--assets", DRIVE_ASSETS)
LOCAL_ROOT = Path("/content/rag_contest_local")
ASSETS = LOCAL_ROOT / "assets" / MODEL
print("คัดลอกและตรวจ hash โมเดลไปพื้นที่ local...")
subprocess.run([PYTHON, "-c", "from rag_contest.runtime import stage_assets; import sys; stage_assets(sys.argv[1], sys.argv[2])",
                str(DRIVE_ASSETS), str(ASSETS)], check=True)
import shutil
if (OUTPUT / "index_cache").is_dir():
    shutil.copytree(OUTPUT / "index_cache", LOCAL_ROOT / "index_cache", dirs_exist_ok=True)
BASE_CONFIG = PROJECT / "config.json"
base_settings = {'device': 'cpu', 'query_mode': 'raw', 'classifier_mode': 'enhanced', 'prompt_mode': 'basic', 'evidence_strategy': 'focused', 'question_order': 'easy_first', 'deferred_retry': True, 'retrieval': 'bm25', 'context_mode': 'full', 'chunk_size': 350, 'overlap': 60, 'evidence_tokens': 1200, 'context_tokens': 5120, 'answer_tokens': 64, 'retry_tokens': 96, 'threads': 2, 'seed': 20261009, 'max_seconds': 5400, 'total_seconds': 6300, 'max_ram_gb': 12.0, 'arithmetic': True}
assert base_settings.get("device", "cpu") == "cpu", "ต้องเป็น CPU"
from rag_contest.pipeline import Config
Config(**base_settings)
BASE_CONFIG.write_text(json.dumps(base_settings))
# These limits apply to fresh inference subprocesses, before native libraries import.
for variable in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[variable] = str(base_settings.get("threads", 2))
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("คง config จากรอบเดิม:", base_settings)
SETUP_SECONDS = time.perf_counter() - SETUP_START
print("อุปกรณ์:", DEVICE, "— GPU ใช้พัฒนาเท่านั้น" if DEVICE == "cuda" else "CPU")
print("โมเดล:", MODEL, "เก็บไว้ที่:", ASSETS)


## 5A. เลือกรอบใหม่หรือรันต่อ
อ่าน RUN_MODE จากข้อ 1 หาก runtime ถูกล้างให้ใช้ resume แล้วรัน 1–5A ก่อน 5B
หากแค่ reconnect และตัวแปรยังอยู่ ให้ใช้ 5B ต่อได้เลย ไม่ต้องรัน 5A
การกด 5A ซ้ำใน runtime เดิมจะใช้ RUN เดิม ไม่สร้างรอบใหม่โดยไม่ตั้งใจ


In [ ]:
import uuid
from rag_contest.data import write_csv, load_questions

def select_run():
    global RUN, EVAL_QUESTIONS, CONFIG
    if RUN_MODE == 'resume':
        chosen = RESUME_PATH
    elif 'RUN' in globals() and (Path(RUN)/'questions.csv').exists():
        chosen = Path(RUN)
        print('ใช้ RUN เดิมใน runtime นี้ ไม่สร้างรอบซ้ำ')
    else:
        # Do not silently start over after a disconnect in a new runtime.
        if LATEST_RUN_FILE.exists():
            previous = Path(json.loads(LATEST_RUN_FILE.read_text())['run'])
            if previous.is_dir() and not (previous/'answer.csv').exists():
                raise RuntimeError(f'พบรอบเดิมที่ยังไม่มี answer.csv: {previous}; ตั้ง RUN_MODE="resume" ในข้อ 1 แล้วรันใหม่ตามลำดับ')
        chosen = OUTPUT/'runs'/('baseline-contest-'+uuid.uuid4().hex[:10])
        chosen.mkdir(parents=True,exist_ok=False)
        write_csv(chosen/'questions.csv', questions, ['question_id','question_text'])
        (chosen/'config.json').write_text(BASE_CONFIG.read_text())
        (chosen/'run_setup.json').write_text(json.dumps({'model':MODEL,'assets':str(ASSETS),
            'question_source':str(QUESTIONS),'setup_seconds':SETUP_SECONDS},indent=2))
        pins = subprocess.check_output([UV,'pip','freeze','--python',PYTHON],text=True)
        (chosen/'environment.start.freeze.txt').write_text(pins)
    assert load_questions(chosen/'questions.csv') == questions, 'คำถามต่างจาก RUN เดิม หยุดก่อนปะปนผล'
    saved = json.loads((chosen/'config.json').read_text())
    assert saved == json.loads(BASE_CONFIG.read_text()), 'config ไม่ตรงกับรอบเดิม ห้ามเปลี่ยนกลางรอบ'
    RUN = chosen
    EVAL_QUESTIONS, CONFIG = RUN/'questions.csv', RUN/'config.json'
    LATEST_RUN_FILE.write_text(json.dumps({'run':str(RUN)},indent=2))
    checkpoint=RUN/'checkpoint.json'
    traces=json.loads(checkpoint.read_text()).get('traces',[]) if checkpoint.exists() else []
    print('RUN:',RUN)
    print('บันทึกแล้ว:',len(traces),'ข้อ; รันต่อเฉพาะงานที่ยังไม่เสร็จ')
    print('ขั้นต่อไป: 5B' if not (RUN/'answer.csv').exists() else 'มี answer.csv แล้ว ไปข้อ 6–7')

select_run()


## 5B. ตอบ / รันต่อ
เก็บ stdout และ stderr ใน run.log อ่าน error จากเซลล์นี้หากยังไม่มี status.json
การรันต่อไม่แก้ข้อ unresolved ที่ระบบลองครบแล้ว และไม่เติมคำตอบสมมติ
หากโปรแกรมแจ้ง fingerprint/environment ไม่ตรง หยุดและส่ง error ห้ามลบ checkpoint
ข้อ unresolved ที่ลองครบแล้วจะไม่ถูกตอบใหม่โดยการ resume

In [ ]:
print('RUN:',RUN,flush=True)
if (RUN/'answer.csv').exists():
    print('มีคำตอบครบแล้ว ไปข้อ 6')
else:
    fresh = not (RUN/'checkpoint.json').exists()
    extra = ['--fresh','--setup-seconds',str(SETUP_SECONDS)] if fresh else []
    command=[PYTHON,'-u','-m','rag_contest','run','--docs',str(DOCS),
        '--questions',str(EVAL_QUESTIONS),'--assets',str(ASSETS),
        '--config',str(CONFIG),'--output',str(RUN),'--expected-count',str(EXPECTED_COUNT),*extra]
    try:
        code = stream_command(command, log_path=RUN/'run.log')
    finally:
        if (LOCAL_ROOT/'index_cache').is_dir():
            shutil.copytree(LOCAL_ROOT/'index_cache',OUTPUT/'index_cache',dirs_exist_ok=True)
    print('รหัสจบ:',code)
    if code not in (0,2):
        raise RuntimeError(f'ขั้นตอบผิดพลาดรหัส {code}; ส่งข้อความท้าย run.log ก่อนรันใหม่')


## 5C. ดูความคืบหน้าหลังจบหรือกดหยุด

In [ ]:
checkpoint = RUN / "checkpoint.json"
traces = json.loads(checkpoint.read_text()).get("traces", []) if checkpoint.exists() else []
print("บันทึกแล้ว:", len(traces), "ข้อ")
print("ผ่านรูปแบบ:", sum(t.get("status") == "ok" for t in traces), "ข้อ (ยังไม่ใช่จำนวนตอบถูก)")
print("ยังตอบไม่สำเร็จ:", [t["question_id"] for t in traces if t.get("status") != "ok"])
print("ไฟล์ส่งครบ:", (RUN / "answer.csv").exists())
print("โฟลเดอร์:", RUN)
if not (RUN / "status.json").exists():
    print("ยังไม่จบรอบ: รัน 5B ต่อหรือตรวจ error ใน 5B")


## 6. ตรวจความครบและดูเวลา
ไม่มีคะแนนความถูกต้องเพราะไม่อ่านเฉลย หากยังไม่มี status.json ให้ตรวจ log ไม่ใช่ข้ามการตรวจ

In [ ]:
status_path=RUN/'status.json'
READY_TO_SUBMIT=False
if not status_path.exists():
    print('ยังไม่มี status.json; RUN:',RUN)
    print('ไฟล์:',[p.name for p in RUN.iterdir()])
    if (RUN/'run.log').exists():
        print('\n'.join((RUN/'run.log').read_text(encoding='utf-8',errors='replace').splitlines()[-40:]))
else:
    status=json.loads(status_path.read_text())
    print('ครบ:',status['complete'],'ยังไม่สำเร็จ:',status.get('unresolved_ids',[]))
    READY_TO_SUBMIT=status['complete'] and (RUN/'answer.csv').exists()
if (RUN/'metrics.json').exists():
    print(json.loads((RUN/'metrics.json').read_text()))
print('พร้อมตรวจไฟล์ส่ง:',READY_TO_SUBMIT)


## 7. ส่งออก answer.csv ได้แม้ตอบไม่ครบ
ใช้คำตอบที่บันทึกและผ่านการตรวจแล้ว ข้อที่ไม่สำเร็จเว้น answer_text และ document_id ว่าง ไม่เติมคำตอบสมมติ
ไฟล์อยู่ RUN/submission/answer.csv ไม่เปลี่ยน checkpoint และไม่ทำให้ระบบเข้าใจว่ารันครบ
รันเซลล์นี้ได้แม้ขั้นตอบหยุด แต่ต้องมี RUN และไฟล์คำถาม ตรวจว่ากติกายอมรับแถวว่างก่อนส่ง

In [ ]:
import csv, json
from pathlib import Path

# อ่านรายการคำถามต้นฉบับตามลำดับ ไม่อ่านเฉลย
question_file = RUN / "questions.csv"
if not question_file.exists():
    question_file = Path(QUESTIONS)
with question_file.open(encoding="utf-8-sig", newline="") as f:
    questions_to_export = list(csv.DictReader(f))
ids = [q["question_id"] for q in questions_to_export]
assert len(ids) == len(set(ids)), "question_id ซ้ำ ต้องตรวจไฟล์คำถาม"

accepted = {}
checkpoint = RUN / "checkpoint.json"
if checkpoint.exists():
    traces = json.loads(checkpoint.read_text(encoding="utf-8")).get("traces", [])
    accepted = {str(t["question_id"]): t for t in traces if t.get("status") == "ok"}
else:
    for name in ("answer.csv", "partial_answers.csv"):
        source = RUN / name
        if source.exists():
            with source.open(encoding="utf-8-sig", newline="") as f:
                accepted = {r["question_id"]: r for r in csv.DictReader(f)}
            break

rows, missing = [], []
for qid in ids:
    item = accepted.get(qid, {})
    answer_text = str(item.get("answer_text") or "")
    document_id = str(item.get("document_id") or "")
    if not answer_text.strip() or not document_id.strip():
        answer_text, document_id = "", ""
        missing.append(qid)
    rows.append({"question_id": qid, "answer_text": answer_text, "document_id": document_id})

# แยกจาก RUN/answer.csv ซึ่งระบบใช้ตัดสินว่ารันครบแล้ว
export_dir = RUN / "submission"
export_dir.mkdir(parents=True, exist_ok=True)
answer = export_dir / "answer.csv"
with answer.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["question_id", "answer_text", "document_id"])
    writer.writeheader()
    writer.writerows(rows)
(export_dir / "export_status.json").write_text(json.dumps({
    "count": len(rows), "answered": len(rows)-len(missing),
    "blank_question_ids": missing, "complete": not missing
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("สร้างไฟล์:", answer)
print("ทั้งหมด:", len(rows), "ข้อ; มีคำตอบ:", len(rows)-len(missing), "ข้อ; ว่าง:", missing)
print("ไฟล์มีครบทุกแถว แต่แถวว่างไม่ใช่คำตอบที่สำเร็จ ตรวจว่ากติกายอมรับช่องว่างก่อนส่ง")
from google.colab import files
files.download(str(answer))
